In [70]:
import sys
import os
sys.path.append(os.path.abspath('..')) 
%load_ext autoreload
%autoreload 2
from src import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [71]:
import torch
import numpy as np
from pathlib import Path
from torch.utils.data import DataLoader, random_split
import time

In [72]:
def set_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    try:
        import intel_extension_for_pytorch as ipex
        if torch.xpu.is_available():
            return torch.device('xpu')
    except ImportError:
        pass
    # Otimizações para CPU Intel Core Ultra
    torch.set_num_threads(os.cpu_count())
    torch.backends.mkldnn.enabled = True
    return torch.device('cpu')

In [ ]:
# Reprodutividade
torch.manual_seed(42)
np.random.seed(42)

# Configurações
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'xpu' if hasattr(torch, 'xpu') and torch.xpu.is_available() else 'cpu')
MODEL = UNetBinary().to(DEVICE)
PATH_DIR = Path('../data/stage1_train')
BATCH_SIZE = 16
TRAIN_SIZE = 0.8
EPOCHS = 10
LR = 1e-4
PART = 1

# Dataset
full_dataset = DSB2018Dataset(root_dir=PATH_DIR, img_size=128, cache_in_memory=True)
train_size = int(TRAIN_SIZE * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# Dataloader
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)


In [74]:
def training(
    model: nn.Module,
    dataloader: DataLoader,
    device: torch.device,
    part: int,
    lr: float = 1e-4,
    num_epochs: int = 10
) -> dict:
    """
    Treina o modelo para parte 1 (binária) ou parte 2 (embeddings).
    Suporte nativo para CUDA, Intel XPU (Arc) e CPU.
    """
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    num_batches = len(dataloader)

    print(f"Treinando em: {device}")
    print(f"Batches por época: {num_batches}")

    history = {'loss': [], 'iou': [], 'dice': []}

    for epoch in range(num_epochs):
        model.train()

        accumulated_loss = torch.tensor(0.0, device=device)
        total_intersection = torch.tensor(0.0, device=device)
        total_union = torch.tensor(0.0, device=device)

        for images, masks, instance_gt in dataloader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            optimizer.zero_grad()

            if part == 1:
                prediction_bin = model(images)
                loss = criterion(prediction_bin, masks)
            # else:
            #     instance_gt = instance_gt.to(device, non_blocking=True)
            #     prediction_bin, prediction_emb = model(images)
            #     loss_bin = criterion(prediction_bin, masks)
            #     loss_emb = discriminative_loss(prediction_emb, instance_gt)
            #     loss = loss_bin + loss_emb

            loss.backward()
            optimizer.step()

            with torch.no_grad():
                binary_prediction = (prediction_bin > 0).float()
                intersection = (binary_prediction * masks).sum()
                union = binary_prediction.sum() + masks.sum() - intersection

                accumulated_loss += loss.detach()
                total_intersection += intersection
                total_union += union

        epoch_loss = accumulated_loss.item() / num_batches
        ti = total_intersection.item()
        tu = total_union.item()

        epoch_iou = ti / (tu + 1e-6)
        epoch_dice = (2.0 * ti) / (tu + ti + 1e-6)

        history['loss'].append(epoch_loss)
        history['iou'].append(epoch_iou)
        history['dice'].append(epoch_dice)

        print(f"Epoch {epoch+1}/{num_epochs} | Loss: {epoch_loss:.4f} | IoU: {epoch_iou:.4f} | Dice: {epoch_dice:.4f}")

    return history

In [75]:
start = time.time()
history = training(
            model=MODEL,
            dataloader=train_loader,
            device=DEVICE,
            part=PART,
            lr=LR,
            num_epochs=EPOCHS
)
print(f'Tempo de treinamento: {(time.time() - start)/60:.2f} minutos')


Treinando em: xpu
Batches por época: 34
Epoch 1/10 | Loss: 0.6759 | IoU: 0.1269 | Dice: 0.2253
Epoch 2/10 | Loss: 0.3595 | IoU: 0.0000 | Dice: 0.0000
Epoch 3/10 | Loss: 0.2830 | IoU: 0.0000 | Dice: 0.0000
Epoch 4/10 | Loss: 0.2258 | IoU: 0.1100 | Dice: 0.1982
Epoch 5/10 | Loss: 0.1805 | IoU: 0.5149 | Dice: 0.6798
Epoch 6/10 | Loss: 0.1495 | IoU: 0.6240 | Dice: 0.7685
Epoch 7/10 | Loss: 0.1258 | IoU: 0.6721 | Dice: 0.8039
Epoch 8/10 | Loss: 0.1159 | IoU: 0.6940 | Dice: 0.8193
Epoch 9/10 | Loss: 0.1091 | IoU: 0.7108 | Dice: 0.8310
Epoch 10/10 | Loss: 0.1049 | IoU: 0.7226 | Dice: 0.8390
Tempo de treinamento: 1.03 minutos
